# 01 — Setup and Data Acquisition


Os ficheiros foram carregados para o volume Databricks:

`/Volumes/main/default/faers_data/`

Esta pasta será usada como fonte de dados brutos. A partir deste ponto, os ficheiros originais não serão modificados.

In [0]:
# criação de caminho base e verificação que os dados foram carregados corretamente para o caminho definido 

faers_base_path = "/Volumes/main/default/faers_data/"

display(dbutils.fs.ls(faers_base_path))

## 1.1 Validação da estrutura da fonte

Após a aquisição, o diretório raw contém as seguintes pastas FAERS: `DEMO`, `DRUG`, `REAC`, `OUTC` e `THER`.

Estas pastas correspondem às principais entidades do dataset FAERS. 

In [0]:
# validação de que todos os dados dos trimestres necessários estão presentes nas pastas correspondentes

for table in ["DEMO", "DRUG", "REAC", "OUTC", "THER"]:
    print(f"\n--- {table} ---")
    display(dbutils.fs.ls(f"{faers_base_path}{table}/"))

## 1.2 Seleção das tabelas FAERS

Para o projeto foram selecionadas inicialmente quatro tabelas principais do dataset FAERS: 

- `DEMO`: contém informação geral do relatório/caso e dados demográficos do paciente;
- `DRUG`: contém os medicamentos associados a cada relatório;
- `REAC`: contém as reações adversas reportadas.
- `OUTC`: contém desfechos clínicos associados aos casos reportados.

Estas quatro tabelas permitem relacionar casos, medicamentos e eventos adversos através de identificadores comuns como `primaryid` e `caseid`.

Nesta fase foi realizada uma leitura exploratória das tabelas selecionadas (`DEMO`, `DRUG`,`REAC` e `OUTC`) apenas para confirmar que os ficheiros raw são interpretados corretamente com o delimitador `$`.

Como o objetivo desta etapa é apenas validar a estrutura e visualizar amostras, os tipos de dados ainda não foram convertidos. O carregamento definitivo para a camada Bronze será feito posteriormente com schemas explícitos.

# 📚 Dicionário de Dados FAERS

Fonte: https://fis.fda.gov/extensions/FPD-QDE-FAERS/FPD-QDE-FAERS.html

## Tabela DEMO (Demographics)

Esta tabela contém informações demográficas e administrativas sobre o paciente e o relatório do evento adverso submetido à FDA.

| Nome da Coluna | Descrição | Exemplos / Observações |
| :--- | :--- | :--- |
| **`primaryid`** | Número exclusivo para identificar uma notificação FAERS. Este é o campo de ligação primária. | Chave concatenada do ID do Caso e do Número da Versão do Caso. |
| **`caseid`** | Número para identificar um caso FAERS. | Identificador único do caso. |
| **`caseversion`** | Número da Versão do Relatório de Segurança. | O Caso Inicial será a versão 1; os acompanhamentos terão números incrementados (ex: 2, 3, 4). |
| **`i_f_code`** | Código para o status inicial ou de acompanhamento do relatório. | `I` (Inicial), `F` (Acompanhamento). |
| **`event_dt`** | Data do Evento. | Data em que a reação/evento adverso efetivamente começou (Formato AAAAMMDD). |
| **`mfr_dt`** | Data do Fabricante. | Data em que a farmacêutica/fabricante tomou conhecimento inicial do evento. |
| **`init_fda_dt`** | Data em que a FDA recebeu a primeira versão (Inicial) do Caso. | Formato AAAAMMDD. |
| **`fda_dt`** | Data em que a FDA recebeu o Caso. | Em versões subsequentes, a data mais recente recebida pelo fabricante. |
| **`rept_cod`** | Código para o tipo de relatório enviado. | `EXP` (Expedido - 15 dias), `PER` (Periódico), `DIR` (Direto), `5DAY` (5 Dias), `30DAY` (30 Dias). |
| **`auth_num`** | Número da Autoridade Reguladora. | Referência interna de uma autoridade, se aplicável. |
| **`mfr_num`** | Número do Fabricante. | Identificador único ou número de controlo interno do fabricante do medicamento. |
| **`mfr_sndr`** | Nome do fabricante ou organização que enviou o relatório. | Nome codificado do fabricante ou nome textual da organização. |
| **`lit_ref`** | Referência Literária. | Preenchido se o caso foi extraído de um artigo científico publicado. |
| **`age`** | Idade do paciente. | Apenas o valor numérico (ex: 45, 6, 12). |
| **`age_cod`** | Código/Unidade de medida da Idade. | `YR` (Anos), `MO` ou `MON` (Meses), `DY` (Dias), `WK` (Semanas), `DEC` (Décadas), `HR` (Horas). |
| **`age_grp`** | Código do Grupo de Idade do Paciente. | `N` (Neonato), `I` (Bebê), `C` (Criança), `T` (Adolescente), `A` (Adulto), `E` (Idoso). |
| **`sex`** | Sexo do paciente. | `M` (Masculino), `F` (Feminino), `UNK` (Desconhecido). |
| **`e_sub`** | Submissão Eletrónica. | Indica se o relatório entrou no sistema de forma eletrónica (`Y` = Sim, `N` = Não, `U` = Desconhecido). |
| **`wt`** | Peso do paciente. | Valor numérico do peso do paciente. |
| **`wt_cod`** | Código/Unidade de medida do Peso do paciente. | `KG` (Quilogramas), `LBS` (Libras), `GMS` (Gramas). Importante para conjugar com a coluna de peso (`wt`). |
| **`rept_dt`** | Data de Reporte à FDA. | Data em que a FDA recebeu o relatório pela primeira vez. |
| **`to_mfr`** | Notificação ao fabricante. | `Y` (Sim) ou `N` (Não) se o notificador voluntário também notificou o fabricante. |
| **`occp_cod`** | Ocupação/Profissão de quem reportou o evento. | `MD` (Médico), `PH` (Farmacêutico), `CN` (Consumidor/Paciente), `OT` (Outro). |
| **`reporter_country`** | País. | País de origem de quem reportou o evento (ex: `US`, `FR`, `PT`). |
| **`occr_country`** | País de Ocorrência. | País onde o evento ocorreu. |

---

## 💊 Tabela DRUG (Informação do Medicamento)

Contém os detalhes de todos os medicamentos que o paciente estava a tomar no momento do evento adverso.

| Nome da Coluna | Descrição | Exemplos / Observações |
| :--- | :--- | :--- |
| **`primaryid`** | Número exclusivo para identificar uma notificação FAERS (Chave Primária). | Chave concatenada do ID do Caso e do Número da Versão do Caso. |
| **`caseid`** | Número para identificar um caso FAERS. | Identificador único do caso. |
| **`drug_seq`** | Sequência do medicamento (Chave Secundária). | Numeração (1, 2, 3...) para distinguir múltiplos medicamentos no mesmo relatório. |
| **`role_cod`** | Papel do medicamento no evento. | `PS` (Suspeito Primário), `SS` (Suspeito Secundário), `C` (Concomitante), `I` (Interativo). |
| **`drugname`** | Nome do medicamento. | Nome comercial validado ou nome textual, exatamente como inserido no relatório. |
| **`prod_ai`** | Princípio ativo (Active Ingredient). | A substância química base do medicamento. |
| **`val_vbm`** | Código da fonte do nome do medicamento (`drugname`). | `1` (Nome comercial validado), `2` (Nome textual usado). |
| **`route`** | Via de administração do medicamento. | Como foi tomado (ex: Oral, Intravenosa, Tópica). |
| **`dose_vbm`** | Texto original (Verbatim) da dose. | Texto exato para dose, frequência e via, conforme inserido no relatório. |
| **`cum_dose_chr`** | Dose cumulativa. | Dose total acumulada até à primeira reação adversa. |
| **`cum_dose_unit`** | Unidade da dose cumulativa. | `KG`, `GM`, `MG`, `UG`, `ML`, `IU`, `PCT` (%), `DF` (Forma de Dosagem), etc. |
| **`dechal`** | Resultado da suspensão do medicamento (Dechallenge). | `Y` (Positivo, a reação diminuiu), `N` (Negativo), `U` (Desconhecido), `D` (Não se aplica). |
| **`rechal`** | Resultado da reintrodução (Rechallenge). | `Y` (Positivo, a reação voltou), `N` (Negativo), `U` (Desconhecido), `D` (Não se aplica). |
| **`lot_num`** | Número do lote do medicamento. | Conforme relatado na notificação. |
| **`exp_dt`** | Data de validade do medicamento. | Formato `AAAAMMDD`. |
| **`nda_num`** | Número NDA (New Drug Application). | Apenas valor numérico. |
| **`dose_amt`** | Quantidade da dose relatada. | Valor numérico da dose. |
| **`dose_unit`** | Unidade da dose do medicamento. | `MG`, `ML`, etc. Conjugar com `dose_amt`. |
| **`dose_form`** | Forma farmacêutica. | Comprimido, Cápsula, Xarope, etc. |
| **`dose_freq`** | Frequência da toma. | `1X` (Uma vez), `BID` (2x/dia), `QD` (Diariamente), `PRN` (Conforme necessário), `UNK` (Desconhecido), etc. |

---

## ⚠️ Tabela REAC (Reações / Eventos Adversos)
Lista os sintomas, reações ou problemas médicos que ocorreram. Um relatório pode ter múltiplas reações.

| Nome da Coluna | Descrição | Exemplos / Observações |
| :--- | :--- | :--- |
| **`primaryid`** | Número exclusivo para identificar uma notificação FAERS (Chave Primária). | Chave concatenada do ID do Caso e do Número da Versão do Caso. |
| **`pt`** | Termo Preferencial (Preferred Term). | O sintoma/reação padronizado segundo o dicionário médico MedDRA (ex: *Nausea*, *Headache*, *Myocardial infarction*). |
| **`drug_rec_act`** | Ação tomada. | O que foi feito em resposta à reação (ex: Dose reduzida, Medicamento suspenso). |

---

## 🏥 Tabela OUTC (Resultados / Desfechos)
Regista o desfecho clínico do paciente em consequência do evento adverso. Um paciente pode ter múltiplos desfechos.

| Nome da Coluna | Descrição | Exemplos / Observações |
| :--- | :--- | :--- |
| **`primaryid`** | Número exclusivo para identificar uma notificação FAERS (Chave Primária). | Chave concatenada do ID do Caso e do Número da Versão do Caso. |
| **`outc_cod`** | Código do Desfecho (Outcome). | **Crucial:** <br> `DE` = Death (Morte) <br> `LT` = Life-Threatening (Risco de Vida) <br> `HO` = Hospitalization (Hospitalização) <br> `DS` = Disability (Incapacidade) <br> `CA` = Congenital Anomaly (Anomalia Congénita) <br> `RI` = Required Intervention (Intervenção Necessária) <br> `OT` = Other (Outro) |

In [0]:
# definição das tabelas a explorar
selected_tables = ["DEMO", "DRUG", "REAC", "OUTC"]

# Gerar uma amostra de cada uma das tabelas
for table in selected_tables:
    print(f"\n--- Amostra da tabela {table} ---")
    
    df_sample = (
        spark.read
        .option("header", "true")
        .option("delimiter", "$")
        .csv(f"{faers_base_path}{table}/")
    )
    print(f"Nº de entradas total: {df_sample.count():,}")
    print(f"Colunas: {len(df_sample.columns)}")
    display(df_sample.limit(5))


## Conclusão do Setup Inicial

A etapa de setup e aquisição confirmou que os dados FAERS necessários para o projeto estão disponíveis em `/Volumes/main/default/faers_data/`.

Foram selecionadas quatro tabelas principais para análise: `DEMO`, `DRUG`, `REAC` e `OUTC`. 
Estas tabelas permitem relacionar relatórios, medicamentos, reações adversas e desfechos clínicos.

A leitura exploratória confirmou que os ficheiros raw são lidos corretamente com cabeçalho e delimitador `$`. Nesta fase, não foram aplicadas transformações aos dados nem definidos schemas para cada tabela. 

# 02 — Bronze Loading


Este notebook cria a camada Bronze do projeto FAERS.

De acordo com a arquitetura medalhão, a camada Bronze corresponde à conversão dos ficheiros raw para tabelas Delta, mantendo os dados o mais próximo possível da origem. Os dados foram lidos a partir de `/Volumes/main/default/faers_data/` e são guardados em `/Volumes/main/default/faers_data/delta/bronze/`.

Nesta etapa são aplicados schemas explícitos e adicionadas colunas de metadata técnica:

- `source_file`: identifica o ficheiro raw de origem de cada registo;
- `load_timestamp`: regista o momento em que os dados foram carregados para Bronze.

Não são aplicadas limpezas, deduplicações ou normalizações nesta camada. Essas operações serão realizadas posteriormente na camada Silver.




###*Estratégia de schema*

Na camada Bronze, todas as colunas vão ser  carregadas como `string` de forma temporária.

Esta decisão preserva os valores originais dos ficheiros raw e evita perdas ou conversões incorretas na primeira etapa do pipeline especialmente em campos como datas, idades, pesos, identificadores e códigos clínicos. Apesar de os tipos de dados serem preservados como texto, o schema continua a ser definido de forma explícita.

*A camada Bronze tem como objetivo materializar os dados de forma próxima do estado puro em formato Delta, com schema explícito e acrescentar dias colunas de metadata.* 

Vamos definir uma função auxiliar de uso único apenas para passar o schema temporário a todas as tabelas para facilitar leitura de código. De seguida foram gravadas todas as tabelas, com a adição de duas colunas para registar metadados, em formato delta de forma automática com recurso a um for_loop

Por fim é feita validação de que todas as tabelas foram guardadas no formato e schema correto e que as colunas criadas foram adicionadas. 



In [0]:
# imports necessários
from pyspark.sql.functions import input_file_name, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
# criação de função auxiliar para definir o schema de uma tabela manualmente

from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, current_timestamp


def create_string_schema(columns):
    return StructType([
        StructField(column, StringType(), True)
        for column in columns
    ])


In [0]:
# criação de listas de colunas para cada uma das tabelas 

demo_columns = [
    "primaryid", "caseid", "caseversion", "i_f_code",
    "event_dt", "mfr_dt", "init_fda_dt", "fda_dt",
    "rept_cod", "auth_num", "mfr_num", "mfr_sndr",
    "lit_ref", "age", "age_cod", "age_grp", "sex", "e_sub",
    "wt", "wt_cod", "rept_dt", "to_mfr",
    "occp_cod", "reporter_country", "occr_country"
]

drug_columns = [
    "primaryid", "caseid", "drug_seq", "role_cod",
    "drugname", "prod_ai", "val_vbm", "route",
    "dose_vbm", "cum_dose_chr", "cum_dose_unit",
    "dechal", "rechal", "lot_num", "exp_dt",
    "nda_num", "dose_amt", "dose_unit",
    "dose_form", "dose_freq"
]

reac_columns = [
    "primaryid", "caseid", "pt", "drug_rec_act"
]

outc_columns = [
    "primaryid", "caseid", "outc_cod"
]


In [0]:
# passagem do schema para cada uma das tabelas

schemas = {
    "DEMO": create_string_schema(demo_columns),
    "DRUG": create_string_schema(drug_columns),
    "REAC": create_string_schema(reac_columns),
    "OUTC": create_string_schema(outc_columns)
}

In [0]:
# carregamento do schema para todas as tabelas e adição de colunas com metadata(source_file, load_timestamp)
bronze_dfs = {}

for table_name in selected_tables:
    input_path = f"{faers_base_path}{table_name}/"
    schema = schemas[table_name]
    
    df = (
        spark.read
        .option("header", "true")
        .option("delimiter", "$")
        .schema(schema)
        .csv(input_path)
        .withColumn("source_file", col("_metadata.file_path")) # Adição de coluna com nome do ficheiro
        .withColumn("load_timestamp", current_timestamp()) # Adição de coluna com timestamp
    )
    
    bronze_dfs[table_name] = df
    
    print(f"\nTabela raw carregada para DataFrame Bronze: {table_name}")
 

In [0]:
# Gravação de todas todas as tabelas em formato Delta 
bronze_delta_path = "/Volumes/main/default/faers_data/delta/bronze"

for table_name, df in bronze_dfs.items():
    output_name = f"{table_name.lower()}"
    delta_path = f"{bronze_delta_path}/{output_name}"

    (
        df
        .write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .save(delta_path)
    )

    print(f"Tabela Delta Bronze criada em: {delta_path}")

In [0]:
# validação final: schema correto, carregamento para formato delta e metadata
from pyspark.sql.functions import count, countDistinct

for table_name in selected_tables:
    delta_path = f"{bronze_delta_path}/{table_name.lower()}/"
    df = spark.read.format("delta").load(delta_path)

    print(f"\nMetadata Bronze: {table_name}")

    df.select(
        count("*").alias("total_rows"),
        count("source_file").alias("rows_with_source_file"),
        count("load_timestamp").alias("rows_with_load_timestamp"),
        countDistinct("source_file").alias("distinct_source_files")
    ).show(truncate=False)
    df.printSchema()
    display(
        df.groupBy("source_file")
          .count()
          .orderBy("source_file")
    )


## Conclusão da camada Bronze

Foram criadas quatro tabelas Bronze em formato Delta para as entidades `DEMO`, `DRUG`, `REAC` e `OUTC`.

A escrita foi feita em modo `overwrite`, uma vez que o projeto trabalha com um período fechado de dados, de `2022Q4` a `2023Q4`, permitindo recriar a camada Bronze de forma reprodutível durante o desenvolvimento.



# 03 — Profiling & Silver Cleaning.

O Data Profiling vai ajudar-nos a entender a qualidade dos dados (nulos, duplicados e distribuição). A Silver Cleaning aplicará as regras de negócio para corrigir esses problemas e ajustar os esquemas (schema casting).

### 3.1 Setup inicial
Nesta fase vamos realizar os imports, carregar as tabelas previamente guardadas em formato Delta e definir os caminhos base de cada uma delas.

In [0]:
# Setup inicial
# imports necessários, definição de caminhos base necessários e lista com as tabelas a trabalhar
from pyspark.sql import functions as F
from pyspark.sql.types import *

silver_delta_path = "/Volumes/main/default/faers_data/delta/silver"

tables = ["demo", "drug", "reac", "outc"]

In [0]:
# Primeiro carregamos as tabelas Delta da camada Bronze
bronze_dfs = {}
for table in tables:
    bronze_dfs[table] = spark.read.format("delta").load(f"{bronze_delta_path}/{table}/")
    

In [0]:
# visualização das tabelas
for table in tables:
    print(f"\nTabela: {table}")
    display(bronze_dfs[table].limit(5))


### 3.2 Profiling inicial
Após verificarmos que as tabelas bronze foram corretamente carregadas vamos começar a fazer um profiling inicial.

Nesta fase queremos perceber a estrutura e qualidade dos dados antes de definir regras de limpeza.

In [0]:
# Faz-se uma contagem do nº de registos e de colunas de cada tabela na fase bronze.
# Neste caso, todas as tabelas foram carregadas com um schema que define todas as colunas como string.
# Ainda assim, é importante realizar uma última verificação.
for table, df in bronze_dfs.items():
    print(f"Tabela {table.upper()} apresenta {bronze_dfs[table].count():,} registos.")
    print(f"Tabela {table.upper()} apresenta {len(bronze_dfs[table].columns)} colunas.")
    print(f"\nSchema da tabela {table.upper()}:")
    df.printSchema()

### 3.3 Verificação de nulos

In [0]:
import pyspark.sql.functions as F

# Iterar sobre cada tabela guardada no dicionário bronze_dfs
for table_name, df in bronze_dfs.items():
    print(f"--- Percentagem de Nulos para a Tabela: {table_name.upper()} ---")
    
    total_rows = df.count()
    
    # Prevenção de erro caso alguma tabela não tenha registos (divisão por zero)
    if total_rows == 0:
        print("A tabela está vazia.\n")
        continue

    null_analysis = df.select(
        *[
            F.round(
                F.sum(
                    F.when(
                        (F.col(c).isNull()) | (F.trim(F.col(c)) == ""),1).otherwise(0)) / total_rows * 100,1
                ).alias(c)
            for c in df.columns
            ]
    )
    
    display(null_analysis)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

for table_name, df in bronze_dfs.items():
    print(f"--- Análise Detalhada de Nulos: {table_name.upper()} ---")
    
    total_rows = df.count()
    
    if total_rows == 0:
        print("A tabela está vazia.\n")
        continue

    # 1. Calcula a contagem de nulos para todas as colunas
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
    null_counts_row = df.select(*null_exprs).collect()[0]
    
    # 2. Constrói a lista APENAS com colunas que têm nulos
    summary_data = []
    for c in df.columns:
        n_nulls = null_counts_row[c]
        
        # Só adiciona à lista se tiver pelo menos 1 nulo
        if n_nulls > 0:
            pct_nulls = round((n_nulls / total_rows) * 100, 2)
            summary_data.append((c, n_nulls, pct_nulls))
            
    # 3. Se a lista estiver vazia (zero nulos na tabela inteira), avisa e avança para a próxima tabela
    if not summary_data:
        print("🎉 Não existem colunas com valores nulos nesta tabela!\n")
        continue
        
    # 4. Define o esquema para o novo DataFrame
    schema = StructType([
        StructField("Nome_Coluna", StringType(), True),
        StructField("Qtd_Nulos", LongType(), True),
        StructField("%_Nulos", DoubleType(), True)
    ])
    
    # 5. Cria o DataFrame, ordena de forma decrescente e exibe
    summary_df = spark.createDataFrame(summary_data, schema)
    summary_df = summary_df.orderBy(F.col("Qtd_Nulos").desc())
    
    display(summary_df)

### 3.3.1 Tratamento de Nulos

O tratamento de valores nulos numa arquitetura Medallion, especialmente ao lidar com dados de saúde do mundo real provenientes de sistemas de reporte voluntário como o FAERS, exige um equilíbrio rigoroso. A simples eliminação de todas as linhas com nulos introduziria um forte viés na amostra, ocultando eventos adversos valiosos.

A estratégia aplicada nesta transição para a camada Silver baseia-se nos seguintes princípios de qualidade de dados:

1. **Integridade Relacional (Drop de Chaves Nulas):** Registos cujas chaves identificadoras (`primaryid` ou `caseid`) sejam nulas são eliminados. Sem estes identificadores, torna-se impossível garantir os *joins* entre tabelas (relacionar corretamente um paciente ao seu medicamento e à respetiva reação adversa), tornando o registo órfão e inútil para o pipeline.
2. **Preservação do Contexto Clínico:** Colunas numéricas contínuas (como `age` ou `wt`) e de datas mantêm os seus valores nulos originais. Em estudos de farmacovigilância, a imputação estatística (usar médias ou medianas) em demografia clínica é perigosa, pois pode mascarar a realidade fenotípica do paciente e falsificar a análise final.
3. **Padronização de Categóricas (FillNA com 'UNK'):** Variáveis categóricas essenciais recebem o valor explícito `"UNK"` (Unknown) ou `"U"`. Isto garante que os modelos analíticos e os *dashboards* downstream contabilizem a "falta de informação" como uma categoria válida e auditável, em vez de lidar com *missing values* imprevisíveis do motor Spark.

In [0]:
# Criação de um novo dicionário para armazenar os DataFrames limpos (Silver)
silver_dfs = {}

for table_name, df in bronze_dfs.items():
    
    # 1. Eliminar registos que não tenham as chaves identificadoras fundamentais
    df_clean = df.dropna(subset=["primaryid", "caseid"])
    
    # 2. Tratamento específico por tabela: Preenchimento de nulos em categóricas
    if table_name == "demo":
        df_clean = df_clean.fillna({
            "sex": "UNK", 
            # "age_grp": "UNK",  <-- Mantido comentado para evitar o erro de coluna não encontrada
            "occp_cod": "UNK", 
            "reporter_country": "UNK",
            "e_sub": "U"
        })
    elif table_name == "drug":
        df_clean = df_clean.fillna({
            "role_cod": "UNK", 
            "route": "UNK",
            "dechal": "U", 
            "rechal": "U",
            "dose_freq": "UNK"
        })
    elif table_name == "reac":
        df_clean = df_clean.fillna({
            "drug_rec_act": "UNK"
        })
        
    silver_dfs[table_name] = df_clean
    
    # Validação do impacto das transformações
    registos_iniciais = df.count()
    registos_finais = df_clean.count()
    registos_removidos = registos_iniciais - registos_finais
    
    print(f"--- Tabela {table_name.upper()} ---")
    print(f"Total de registos ANTES do tratamento: {registos_iniciais:,}")
    print(f"Registos removidos (falta de primaryid/caseid): {registos_removidos:,}")
    print(f"Total de registos APÓS tratamento: {registos_finais:,}\n")

# Atualizar o dicionário principal para as próximas etapas (ex: duplicados) usarem os dados já limpos de nulos críticos
bronze_dfs = silver_dfs

### 3.4 Verificação de duplicados
A análise de duplicados será usada para definir a estratégia de limpeza na camada Silver.
- 
- Na camada Silver serão removidos duplicados exatos e serão mantidos os identificadores necessários para preservar relações entre tabelas. *A deduplicação por chave lógica será aplicada com cuidado, uma vez que algumas tabelas FAERS podem conter múltiplos registos válidos por caso, medicamento, reação ou desfecho.*

In [0]:
for table, df in bronze_dfs.items():
    total_rows = df.count()
    distinct_rows = df.distinct().count()
    duplicate_rows = total_rows - distinct_rows
    
    print(f"{table.upper()}")
    print(f"Total: {total_rows}")
    print(f"Distintos: {distinct_rows}")
    print(f"Duplicados exatos: {duplicate_rows}")
    print("-" * 40)

### 3.4.1 Tratamento de Duplicados

O profiling revelou a presença de duplicados exatos, com particular incidência na tabela `REAC` (mais de 100 mil registos). No contexto do FAERS, isto ocorre frequentemente devido a redundâncias no preenchimento do formulário original ou em submissões de acompanhamento (*follow-ups*) onde os mesmos sintomas são recarregados.

**Estratégia de Limpeza:**
Uma vez que são duplicados exatos (todas as colunas contêm os mesmos valores), estes registos não acrescentam qualquer contexto clínico novo. Pelo contrário, mantê-los causaria enviesamento e dupla contagem (*double-counting*) na fase de modelação ou na criação de dashboards. 

Aplica-se a função `dropDuplicates()` a todas as tabelas para garantir a integridade da camada Silver, mantendo apenas registos únicos para cada combinação de caso, medicamento e reação.

In [0]:
print("=== Tratamento De Duplicados Exatos===\n")

# Dicionário temporário para guardar os DataFrames sem duplicados
dedup_dfs = {}

for table_name, df in bronze_dfs.items():
    # 1. Contagem inicial (antes da remoção)
    total_rows_antes = df.count()
    
    # 2. Remover duplicados exatos (avalia todas as colunas por defeito)
    df_dedup = df.dropDuplicates()
    
    # 3. Contagem final e cálculo da diferença
    total_rows_depois = df_dedup.count()
    duplicados_removidos = total_rows_antes - total_rows_depois
    
    # 4. Guardar o DataFrame limpo no novo dicionário
    dedup_dfs[table_name] = df_dedup
    
    # Mostrar resultados
    print(f"--- Tabela: {table_name.upper()} ---")
    print(f"Total antes: {total_rows_antes:,}")
    print(f"Duplicados removidos: {duplicados_removidos:,}")
    print(f"Total depois: {total_rows_depois:,}\n")

# Atualizar o dicionário principal com os dados agora sem nulos críticos e sem duplicados
bronze_dfs = dedup_dfs

### 3.5 Schema Casting (Conversão de Tipos de Dados)

Na camada Bronze, todas as colunas foram ingeridas temporariamente como `string` para garantir a fidelidade aos ficheiros originais. Agora, na camada Silver, é necessário atribuir os tipos de dados semânticos corretos (Datas, Números Inteiros e Decimais).

**Principais Transformações:**
1. **Datas:** Os ficheiros FAERS utilizam o formato `AAAAMMDD`. Colunas como `event_dt` ou `fda_dt` serão convertidas para `DateType`.*
2. **Métricas Clínicas e Doses:** Colunas quantitativas como `age` (idade), `wt` (peso) e `dose_amt` (quantidade da dose) serão convertidas para `DoubleType` para permitir agregações matemáticas (médias, distribuições) na camada Gold.
3. As variáveis categóricas e identificadores (como `primaryid`, `pt`, `outc_cod`) mantêm-se como `string`.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

print("=== SCHEMA CASTING E GRAVAÇÃO SILVER ===\n")

# Dicionário para guardar as tabelas finais da camada Silver
silver_final_dfs = {}

for table_name, df in bronze_dfs.items():
    df_cast = df
    
    # Transformações específicas para a tabela DEMO
    if table_name == "demo":
        df_cast = df_cast.withColumn("event_dt", F.to_date(F.col("event_dt"), "yyyyMMdd")) \
                         .withColumn("mfr_dt", F.to_date(F.col("mfr_dt"), "yyyyMMdd")) \
                         .withColumn("init_fda_dt", F.to_date(F.col("init_fda_dt"), "yyyyMMdd")) \
                         .withColumn("fda_dt", F.to_date(F.col("fda_dt"), "yyyyMMdd")) \
                         .withColumn("rept_dt", F.to_date(F.col("rept_dt"), "yyyyMMdd")) \
                         .withColumn("age", F.col("age").cast(DoubleType())) \
                         .withColumn("wt", F.col("wt").cast(DoubleType()))
                         
    # Transformações específicas para a tabela DRUG
    elif table_name == "drug":
        df_cast = df_cast.withColumn("exp_dt", F.to_date(F.col("exp_dt"), "yyyyMMdd")) \
                         .withColumn("dose_amt", F.col("dose_amt").cast(DoubleType())) \
                         .withColumn("cum_dose_chr", F.col("cum_dose_chr").cast(DoubleType()))
                         
    # As tabelas REAC e OUTC contêm apenas identificadores e códigos em texto, 
    # pelo que não necessitam de casting numérico/temporal.
    
    silver_final_dfs[table_name] = df_cast
    print(f"Schema atualizado para a tabela: {table_name.upper()}")

print("\n--- A Iniciar Gravação na Camada Silver ---")

# Gravação em formato Delta
for table_name, df in silver_final_dfs.items():
    output_path = f"{silver_delta_path}/{table_name}"
    
    (
        df.write
          .mode("overwrite")
          .format("delta")
          .option("overwriteSchema", "true")
          .save(output_path)
    )
    print(f"✅ Tabela {table_name.upper()} gravada com sucesso em: {output_path}")

# Atualiza os dados em memória para verificação se necessário
silver_dfs = silver_final_dfs

### 3.6 Validação Final da Camada Silver (Data Quality Check)

Antes de darmos a camada Silver como concluída, realizamos uma auditoria final diretamente nos ficheiros Delta que foram gravados no Unity Catalog/Volume. 

Esta validação garante que:
1. O motor Spark consegue ler as tabelas gravadas sem corrupção.
2. O **Schema Casting** foi persistido corretamente (verificando os tipos `date` e `double`).
3. Visualizamos uma amostra real dos dados já limpos de nulos críticos, sem duplicados e com a tipagem correta, prontos para alimentar a camada Gold.

In [0]:
print("=== AUDITORIA E VALIDAÇÃO DA CAMADA SILVER (DELTA) ===\n")

for table in tables:
    silver_path = f"{silver_delta_path}/{table}"
    
    print(f" Matriz de Validação para a tabela: {table.upper()}")
    
    # Ler diretamente do caminho Delta gravado
    df_silver = spark.read.format("delta").load(silver_path)
    
    # 1. Contagem total de linhas salvas
    total_rows = df_silver.count()
    print(f"   -> Total de registos persistidos: {total_rows:,}")
    print(f"   -> Total de colunas: {len(df_silver.columns)}")
    
    # 2. Print do Schema para validar visualmente o Casting
    print("   -> Estrutura do Schema:")
    df_silver.printSchema()
    
    # 3. Mostrar uma amostra rápida dos dados limpos
    print(f"   -> Amostra dos primeiros 3 registos de {table.upper()}:")
    display(df_silver.limit(3))
    
    print("-" * 80)

#### 3.4.2 Duplicados por chaves lógicas
Nas tabelas FAERS, algumas colunas funcionam como identificadores importantes. Para este projeto, devemos analisar sobretudo :
| Tabela | Colunas relevantes                   |
| ------ | ------------------------------------ |
| DEMO   | `primaryid`, `caseid`, `caseversion` |
| DRUG   | `primaryid`, `caseid`, `drug_seq`    |
| REAC   | `primaryid`, `caseid`, `pt`          |
| OUTC   | `primaryid`, `caseid`, `outc_cod`    |

In [0]:
demo_df = bronze_dfs["demo"]

display(
     demo_df
     .groupBy("primaryid", "caseid", "caseversion")
     .count()
     .filter(F.col("count") > 1)
     .orderBy(F.desc("count"))
      .limit(5)
 )

In [0]:
drug_df = bronze_dfs["drug"]
 
display(
     drug_df
     .groupBy("primaryid", "caseid", "drug_seq")
     .count()
     .filter(F.col("count") > 1)
     .orderBy(F.desc("count"))
     .limit(5)
)

In [0]:
reac_df = bronze_dfs["reac"]
 
display(
     reac_df
     .groupBy("primaryid", "caseid", "pt")
     .count()
     .filter(F.col("count") > 1)
     .orderBy(F.desc("count"))
     .limit(5)
 )

In [0]:
outc_df = bronze_dfs["outc"]
 
display(
     outc_df
     .groupBy("primaryid", "caseid", "outc_cod")
     .count()
     .filter(F.col("count") > 1)
     .orderBy(F.desc("count"))
     .limit(5)
 )